# Exercise 02 — SOLUTION: PGD Attack — Untargeted and Targeted

## Background

See student notebook for theory.

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# Load pretrained ResNet20 trained on CIFAR-10, set to eval mode
model = torch.hub.load('chenyaofo/pytorch-cifar-models', 'cifar10_resnet20', pretrained=True)
model.eval()

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616])
])
testset = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform)
images, labels = next(iter(torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)))
CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

IMG_MIN, IMG_MAX = images.min().item(), images.max().item()

def accuracy(model, x, y):
    with torch.no_grad():
        preds = model(x).argmax(dim=1)
    return (preds == y).float().mean().item()

print(f"Clean accuracy: {accuracy(model, images, labels):.2%}")

## Untargeted PGD (pre-written)

In [ ]:
class PGD:
    """Untargeted PGD attack (pre-written)."""
    def __init__(self, model, epsilon=0.05, alpha=0.005, steps=20):
        self.model, self.epsilon, self.alpha, self.steps = model, epsilon, alpha, steps

    def attack(self, images, labels):
        x = images.clone().detach()
        x = x + torch.empty_like(x).uniform_(-self.epsilon, self.epsilon)
        x = torch.clamp(x, IMG_MIN, IMG_MAX)
        for _ in range(self.steps):
            x = x.detach().requires_grad_(True)
            loss = nn.CrossEntropyLoss()(self.model(x), labels)
            loss.backward()
            with torch.no_grad():
                x = x + self.alpha * x.grad.sign()
                delta = torch.clamp(x - images, -self.epsilon, self.epsilon)
                x = torch.clamp(images + delta, IMG_MIN, IMG_MAX)
        return x.detach()

pgd = PGD(model)
x_adv_untargeted = pgd.attack(images, labels)
print(f"After untargeted PGD: accuracy = {accuracy(model, x_adv_untargeted, labels):.2%}")


## Targeted PGD — Solution

In [ ]:
class PGD_Targeted(PGD):
    """Targeted PGD: force prediction to match `targets`."""

    def attack(self, images, targets):
        # SOLUTION
        x = images.clone().detach()
        x = x + torch.empty_like(x).uniform_(-self.epsilon, self.epsilon)
        x = torch.clamp(x, IMG_MIN, IMG_MAX)
        for _ in range(self.steps):
            x = x.detach().requires_grad_(True)
            loss = nn.CrossEntropyLoss()(self.model(x), targets)
            loss.backward()
            with torch.no_grad():
                x = x - self.alpha * x.grad.sign()  # MINUS: descend toward target
                delta = torch.clamp(x - images, -self.epsilon, self.epsilon)
                x = torch.clamp(images + delta, IMG_MIN, IMG_MAX)
        return x.detach()

targets = (labels + 3) % 10
pgd_t = PGD_Targeted(model, epsilon=0.05, alpha=0.005, steps=40)
x_adv_targeted = pgd_t.attack(images, targets)
with torch.no_grad():
    preds_targeted = model(x_adv_targeted).argmax(1)
target_success = (preds_targeted == targets).float().mean()
print(f"Targeted attack success rate: {target_success:.2%}")


## Compare FGSM vs PGD

In [ ]:
# Compare FGSM vs PGD accuracy across epsilon values
epsilons = [0.01, 0.02, 0.05, 0.1, 0.15]
fgsm_accs, pgd_accs = [], []

for eps in epsilons:
    # FGSM (1-step PGD without random start)
    x_f = images.clone().detach().requires_grad_(True)
    nn.CrossEntropyLoss()(model(x_f), labels).backward()
    x_fgsm = torch.clamp(images + eps * x_f.grad.sign(), IMG_MIN, IMG_MAX).detach()
    fgsm_accs.append(accuracy(model, x_fgsm, labels))

    # PGD
    pgd_accs.append(accuracy(model, PGD(model, epsilon=eps).attack(images, labels), labels))

plt.figure(figsize=(7, 4))
plt.plot(epsilons, fgsm_accs, 'o-', label='FGSM (1-step)')
plt.plot(epsilons, pgd_accs,  's-', label='PGD (20-step)')
plt.xlabel('Epsilon'); plt.ylabel('Accuracy'); plt.title('FGSM vs PGD Attack Strength')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()


## Visualization

In [ ]:
# Visualize targeted attack results
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i in range(5):
    img_show = images[i].permute(1,2,0).numpy()
    img_show = (img_show - img_show.min()) / (img_show.max() - img_show.min())
    adv_show = x_adv_targeted[i].permute(1,2,0).numpy()
    adv_show = (adv_show - adv_show.min()) / (adv_show.max() - adv_show.min())
    with torch.no_grad():
        orig_pred = CIFAR10_CLASSES[model(images[i:i+1]).argmax(1).item()]
        adv_pred  = CIFAR10_CLASSES[model(x_adv_targeted[i:i+1]).argmax(1).item()]
        tgt_cls   = CIFAR10_CLASSES[targets[i].item()]
    axes[0,i].imshow(img_show); axes[0,i].set_title(f'Original\n{orig_pred}', fontsize=9); axes[0,i].axis('off')
    axes[1,i].imshow(adv_show); axes[1,i].set_title(f'Adversarial\npred:{adv_pred}\ntgt:{tgt_cls}', fontsize=8); axes[1,i].axis('off')
plt.suptitle('Targeted PGD Attack', fontsize=13); plt.tight_layout(); plt.show()
